# ECG Anomaly Detection - Data Exploration


In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
import numpy as np
import matplotlib.pyplot as plt
import wfdb
from config import DATA_RAW_DIR, DATA_PROCESSED_DIR, BEAT_CLASSES


## Plot a raw ECG record

In [ ]:
record = wfdb.rdrecord(os.path.join(DATA_RAW_DIR, '100'))
ann = wfdb.rdann(os.path.join(DATA_RAW_DIR, '100'), 'atr')

fig, axes = plt.subplots(2, 1, figsize=(16, 6), sharex=True)
t = np.arange(3600) / 360.0  # first 10 seconds
for i, ax in enumerate(axes):
    ax.plot(t, record.p_signal[:3600, i], lw=0.7)
    ax.set_ylabel(record.sig_name[i])
    ax.set_xlabel('Time (s)')
axes[0].set_title('MIT-BIH Record 100 - First 10 seconds')
plt.tight_layout()
plt.show()


## Class Distribution

In [ ]:
y = np.load(os.path.join(DATA_PROCESSED_DIR, 'y.npy'))
class_names = {v: k for k, v in BEAT_CLASSES.items()}
counts = {class_names[c]: (y == c).sum() for c in range(len(BEAT_CLASSES))}

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(counts.keys(), counts.values(), color='steelblue')
ax.set_title('Beat Class Distribution')
ax.set_ylabel('Count')
for i, (k, v) in enumerate(counts.items()):
    ax.text(i, v + 100, str(v), ha='center')
plt.tight_layout()
plt.show()


## Sample Beats per Class

In [ ]:
X = np.load(os.path.join(DATA_PROCESSED_DIR, 'X.npy'))
t = np.arange(X.shape[1]) / 360.0

fig, axes = plt.subplots(1, len(BEAT_CLASSES), figsize=(16, 3))
for ax, (cls_name, cls_idx) in zip(axes, sorted(BEAT_CLASSES.items(), key=lambda x: x[1])):
    idxs = np.where(y == cls_idx)[0]
    for i in range(min(5, len(idxs))):
        ax.plot(t, X[idxs[i]], alpha=0.6, lw=0.8)
    ax.set_title(f'Class {cls_idx}\n({cls_name})')
    ax.set_xlabel('s')
plt.suptitle('Sample Beats per Class')
plt.tight_layout()
plt.show()


## Results - View plots after training

In [ ]:
from IPython.display import Image, display
import glob

plots_dir = os.path.join('..', 'results', 'plots')
for p in sorted(glob.glob(os.path.join(plots_dir, '*.png'))):
    print(p)
    display(Image(p))
